# Onde ganhar votos para Lula exige menos esforço: o 2º turno de 2026 por bairro

**Estel Tecnologia** ([estel.tec.br](https://estel.tec.br)) · análise de 07 e 08/10/2026

> **Produção independente e exclusiva da Estel Tecnologia (estel.tec.br).** Esta ferramenta não tem relação com a campanha oficial de Lula, com o PT ou com qualquer partido, federação, coligação ou candidatura, e não foi encomendada nem paga por eles.

Esta análise leva ao nível de bairro a pergunta da versão municipal (v2.0, tag `v2.0`): **onde a campanha
deve gastar tempo e dinheiro entre os turnos?** Agora, com a pergunta mais fina: **onde cada voto a mais
para Lula custa menos esforço?**

> **Aviso de método.** Tudo aqui vem de dados agregados: resultado por local de votação (TSE) e perfil
> por setor censitário (IBGE). Nada descreve pessoas. O eleitor vota onde está a escola-sede da seção, não
> necessariamente onde mora. Quanto menor a área, maior o risco da **falácia ecológica**: o padrão de um
> bairro não diz nada sobre cada morador. As projeções supõem que o comportamento de 2022 entre os turnos se
> repete. Não é previsão, e não há "chance de vitória" em lugar nenhum deste documento.

Variante deste notebook: unidade **U4**, método **MC**, pesos **P2** (ver seção 9).

## Resumo executivo

**0. Leia os intervalos como mínimos.** No backtesting entre eleições, o modelo errou mais do que os intervalos
indicavam (2018 → 2022: cobertura de 42%; 2014 → 2018: pior que o swing uniforme). Ele vale enquanto o comportamento
entre os turnos se parece com o de 2022.

**1. A geografia do voto por bairro está estável, o que torna o exercício possível.** Entre o 1º turno de
2022 e o de 2026, a fatia de Lula nas 18.863 áreas tem correlação de
0,977, e a queda variou pouco de área para área (desvio de 4,0
pontos em torno da média de -4,5).

**2. O ponto de partida continua ruim.** Repetido o comportamento de 2022 entre os turnos, a margem nacional
projetada é de -6.610 mil (90%: -6.670 mil a -6.552 mil) votos para Flávio. Com a terceira via mais à direita, -8.307 mil (90%: -9.830 mil a -6.767 mil).

**3. O potencial geográfico por bairro é da mesma ordem que o municipal.** Nos 200 municípios-foco da v2.0, as
áreas somam 503 mil votos de potencial, contra 459 mil no modelo municipal. O
ganho vem quase todo do Nordeste urbano (121 mil, contra
89 mil), onde a queda de Lula variou muito dentro das cidades.

**4. Os 7 níveis de esforço dependem dos pesos e do método.** Entre o esquema de pesos iguais e o de foco em
potencial, 13.089 de 18.566 áreas mudam de nível. No topo, cerca de 2/3
das áreas de nível 7 se mantêm. O componente frágil é a probabilidade de a mobilização render, que muda bastante
entre MCMC e ADVI. Use os níveis como triagem, não como ranking fino.

**5. Pesquisas e voto real divergem no bairro.** Levadas ao bairro por MRP (sexo, idade e cor), as pesquisas dão a
Lula 3,5 pontos a mais que o modelo ecológico, em média, com divergência acima de
10 pontos em 51% das áreas: mais no Sul e Sudeste, menos no Nordeste. O
modelo ecológico, ancorado no voto, é a base; o MRP é contraste.

## 1. Desenho: as variantes

A tarefa foi rodada em modo autônomo. Toda dúvida com mais de um caminho razoável virou variante, registrada em
e comparada na seção 9:

| Eixo | Variantes |
|---|---|
| Unidade de análise | U1 bairros do IBGE (com recuo a subdistrito, distrito e município) · U2 áreas de influência dos locais de votação · U3 malhas das prefeituras + U1 |
| Método de inferência | MC: MCMC (NUTS, nutpie) por UF · M2: ADVI do país inteiro |
| Pesos do índice | P1 iguais · P2 foco em potencial · P3 foco em segurança da mobilização |
| Pesquisas (MRP) | R1 Datafolha · R2 AtlasIntel · R3 combinada |

## 2. Dados e unidade de análise

O TSE não publica resultado por bairro. A unidade é construída: **seção → local de votação → área → município →
região metropolitana → UF**. O TSE publica a latitude e a longitude de cada local de votação, e cada local vai para o
setor censitário do Censo 2022 onde cai a coordenada.

In [1]:
import json, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
m = json.load(open("variantes/U4/medidas.json"))
pd.Series(m["geocodificacao_2026_pct_eleitores"], name="% dos eleitores (2026)").to_frame()

,% dos eleitores (2026)
municipio,0.47
nome_bairro,0.10
outro_ano,0.08
tse_ok,99.35


In [2]:
pd.read_csv("variantes/comparacoes/unidades.csv")[["variante", "unidades_abaixo_do_municipio",
    "cobertura_pct_eleitores_abaixo_do_municipio", "eleitores_por_unidade_mediana", "unidades_fundidas_por_tamanho"]]

,variante,unidades_abaixo_do_municipio,cobertura_pct_eleitores_abaixo_do_municipio,eleitores_por_unidade_mediana,unidades_fundidas_por_tamanho
0,U1,13987,75.89,4102.0,3294
1,U2,12348,83.47,9106.0,911
2,U3,14056,75.88,4086.0,3295
3,U4,15988,89.42,4430.0,3482


Leitura: a coordenada do TSE coloca 99,3% dos eleitores de 2026 num
setor do município certo. O bairro que o TSE declara bate com o bairro do IBGE do ponto em
72% dos locais (nomes populares diferem dos oficiais; a discordância
não prova erro). Áreas com menos de 1.000 eleitores foram fundidas à vizinha mais próxima do mesmo município.

## 3. Série histórica

| Ano | Menor nível publicado | Levado ao bairro? |
|---|---|---|
| 1994 | município e zona | não |
| 1998 | seção, sem local de votação (e sem 2º turno) | não |
| 2002, 2006 | seção e nº do local, sem cadastro com coordenadas | não |
| 2010, 2014, 2018, 2022, 2026 | seção e local com coordenadas | **sim** |

**A série no nível de bairro começa em 2010.** Antes disso, não há como saber onde ficava cada local de votação
sem inventar dado.

In [3]:
pd.read_csv("variantes/U4/historico/estabilidade.csv")

,de,para,unidades,correlacao,variacao_media_pp,desvio_da_variacao_pp
0,2010 T2,2014 T2,18962,0.893,-2.66,8.81
1,2014 T2,2018 T2,18790,0.895,-7.05,10.19
2,2018 T2,2022 T2,18282,0.965,3.97,6.64
3,2010 T1,2014 T1,18962,0.896,-3.90,9.09
4,2014 T1,2018 T1,18790,0.858,-13.47,11.40
5,2018 T1,2022 T1,18282,0.889,16.45,10.07
6,2022 T1,2026 T1,18863,0.977,-4.46,4.03


In [4]:
pd.read_csv("variantes/U4/historico/transferencias.csv")

,ano,pt_1t,pt_2t,adv_1t,adv_2t,terceiros_1t,ganho_pt_por_voto_terceiro,ganho_adv_por_voto_terceiro
0,2010,47603980,55696824,33080322,43638287,20764029,0.390,0.508
1,2014,43271552,54494396,34837207,50944573,25818184,0.435,0.624
2,2018,31351252,47014387,49167915,57671946,26376905,0.594,0.322
3,2022,57166397,60236417,50962432,58073700,9867805,0.311,0.721


## 4. O modelo

O mesmo RxC hierárquico da v2.0 (Rosen, Jiang, King e Tanner, 2001): quatro origens no 1º turno (Lula, adversário,
terceira via, fora) e três destinos no 2º (Lula, adversário, fora), com verossimilhança Dirichlet. A hierarquia
desce ao bairro: **país → UF → região metropolitana → município → área**. As taxas nacionais por faixa do modelo
municipal entram como priori de cada UF (Bayes empírico). RM e município têm dois efeitos (um por destino), e a área,
dois, só onde o município tem mais de uma. Mais que isso não é identificável com dados agregados.

In [5]:
import glob
diag = pd.DataFrame([json.load(open(f)) for f in glob.glob("variantes/U4/MC/diag_??.json")])
diag[["uf", "unidades", "municipios", "segundos", "rhat_max", "ess_min", "divergencias", "descartadas"]].sort_values("rhat_max")

,uf,unidades,municipios,segundos,rhat_max,ess_min,divergencias,descartadas
22,RO,251,52,195,1.005,652,0,[]
26,TO,164,139,250,1.006,448,0,[]
20,SE,235,75,203,1.007,598,4,[]
25,AL,241,102,339,1.008,590,1,[]
2,MT,473,141,480,1.012,486,0,[]
7,AP,86,16,41,1.014,314,13,[]
9,MS,264,79,230,1.015,648,0,[]
6,PR,1158,399,1764,1.021,265,0,[]
13,PE,839,185,2634,1.023,261,0,[]
18,DF,33,1,49,1.024,213,1,[]


**Convergência.** As quantidades que entram nas projeções (fração para Lula entre quem votou, saldo líquido)
convergem melhor que os parâmetros. O que mistura mal é o nível de comparecimento da terceira via, pouco
identificado. As UFs com R-hat acima de 1,05 aparecem sinalizadas, e BA e CE tiveram cadeias em modos separados;
por isso essas UFs receberam um reajuste longo e, onde ele não resolveu, o selo de estimativa menos segura.

## 5. Validação

Duas validações: (a) **fora da amostra**, como na v2.0: ajuste em 80% das áreas e cobertura do intervalo de 90% nas
outras 20%; (b) **backtesting**: ajuste das transferências de uma eleição e previsão do 2º turno da seguinte a partir
do 1º turno dela, área por área, contra o swing uniforme.

In [6]:
import os
v = [json.load(open(f)) for f in glob.glob("variantes/U4/MC/diag_??_val.json")]
print("validação 80/20:", "pendente" if not v else f"{len(v)} UFs ajustadas")
p = "variantes/U4/MC/validacao.csv"
pd.read_csv(p) if os.path.exists(p) else None

validação 80/20: pendente


,faixa,areas,cobertura_ic90,erro_mediano_pp
0,<35%,858.0,0.948718,0.641200
1,35-45%,838.0,0.971360,0.672154
2,45-55%,715.0,0.946853,0.847942
3,55-65%,513.0,0.972710,0.779979
4,>65%,1067.0,0.952202,0.914551
5,TODAS,3991.0,0.957154,0.773510


In [7]:
p = "variantes/U4/historico/backtest.csv"
pd.read_csv(p) if os.path.exists(p) else print("backtesting pendente")

,ajuste,previsto,ufs,unidades,cobertura_ic90,erro_mediano_pp_modelo,erro_mediano_pp_swing_uniforme,margem_total_obs_mi,margem_total_modelo_mi,margem_total_swing_mi
0,2018,2022,27,17566,0.415,3.88,7.64,2.13,8.54,13.77
1,2014,2018,27,17563,0.257,7.39,6.55,-10.68,-24.31,-22.87


## 6. Resultados

Nacional (10.000 sorteios de Monte Carlo, na GPU):

| Medida | Mediana e intervalo de 90% |
|---|---|
| Margem projetada, cenário base | -6.610 mil (90%: -6.670 mil a -6.552 mil) |
| Margem, terceira via mais à direita | -8.307 mil (90%: -9.830 mil a -6.767 mil) |
| Potencial (terreno perdido + 2 pp onde a mobilização rende) | 1.608 mil (90%: 1.556 mil a 1.659 mil) |
| Saldo de +2 pp de comparecimento em todas as áreas | -1.053 mil (90%: -1.144 mil a -958 mil) |

O saldo nacional de mobilizar em todo lugar é negativo: em muitas áreas, quem passa a votar vota mais em Flávio.
Mobilização precisa de endereço.

In [8]:
fr = pd.read_csv("variantes/comparacoes/frentes.csv")
fr[fr.variante == "U4/MC"].set_index("frente")[["areas", "pot_p50", "pot_v2_municipal", "areas_nivel7_P1"]].round(0)

,areas,pot_p50,pot_v2_municipal,areas_nivel7_P1
frente,,,,
Goiás e entorno,200.0,106392.0,105629.0,110.0
Interior de SP,110.0,16651.0,15548.0,4.0
Metropolitana de SP,399.0,114920.0,109525.0,28.0
Minas Gerais,373.0,56295.0,52105.0,32.0
Nordeste urbano,635.0,120527.0,89307.0,281.0
Norte e demais,90.0,20911.0,20531.0,35.0
Sul,585.0,67140.0,66168.0,36.0
TODOS OS FOCOS,2392.0,502838.0,458924.0,526.0


In [9]:
es = pd.read_parquet("variantes/U4/MC/esforco.parquet")
mc = pd.read_parquet("variantes/U4/MC/unidades_mc.parquet")
t = es.merge(mc[["id_unidade", "terreno", "mob_p50", "lula_pct_26", "desloc"]], on="id_unidade")
top = t[t["nivel_P2"] == 7].sort_values("pot_p50", ascending=False)
top[["uf", "municipio", "nome", "tipo", "aptos", "pot_p05", "pot_p50", "pot_p95", "pot_mil", "prob_mob_rende", "nivel_P1", "nivel_P2", "nivel_P3"]].head(30).round(2)

,uf,municipio,nome,tipo,aptos,pot_p05,pot_p50,pot_p95,pot_mil,prob_mob_rende,nivel_P1,nivel_P2,nivel_P3
18981,GO,Aparecida de Goiânia,Cidade Satélite São Luiz e arredores,subdistrito,206648.0,6056.569824,6146.890137,6235.089844,29.75,0.00,7,7,5
19092,GO,Rio Verde,Residencial Monte Sião e arredores,subdistrito,150091.0,5918.270020,5983.549805,6047.290039,39.87,0.00,7,7,5
18980,GO,Aparecida de Goiânia,Jardim das Acácias e arredores,subdistrito,134580.0,5468.259766,5525.009766,5580.439941,41.05,0.00,7,7,5
17773,SP,São Paulo,Jardim Aracati e arredores,subdistrito,238968.0,3991.679932,5433.720215,6788.729980,22.74,0.65,7,7,6
18985,GO,Catalão,Residencial Alvino Albino e arredores,subdistrito,72021.0,4210.979980,4241.669922,4271.629883,58.89,0.00,7,7,6
266,MA,Balsas,Centro e entorno,area_local,67669.0,3740.510010,3770.570068,3799.929932,55.72,0.03,7,7,7
16401,MG,Conselheiro Lafaiete,N.Sra. da Guia e arredores,subdistrito,91708.0,3317.560059,3358.300049,3398.090088,36.62,0.00,7,7,5
19000,GO,Goianésia,Residencial Laurentino Martins e arredores,subdistrito,59437.0,2965.439941,2990.540039,3015.060059,50.31,0.00,7,7,5
18996,GO,Formosa,Setor Bela Vista e arredores,subdistrito,72791.0,2490.469971,2521.909912,2552.600098,34.65,0.00,7,7,5
17785,SP,São Paulo,Jardim Almeida e arredores,subdistrito,114000.0,1710.839966,2377.729980,3045.409912,20.86,0.54,7,7,5


In [10]:
# distribuição dos níveis por frente (pesos P2)
pd.crosstab(t["frente"], t["nivel_P2"], margins=True)

nivel_P2,1,2,3,4,5,6,7,All
frente,,,,,,,,
Goiás e entorno,193,173,155,154,140,160,382,1357
Interior de SP,354,408,282,308,242,145,37,1776
Metropolitana de SP,52,91,105,134,98,61,15,556
Minas Gerais,346,350,257,258,313,373,337,2234
Nordeste urbano,107,293,744,910,1153,1155,1307,5669
Norte e demais,375,375,439,405,153,112,160,2019
Rio de Janeiro,188,379,301,105,52,31,10,1066
Sul,1038,583,369,378,501,615,405,3889
All,2653,2652,2652,2652,2652,2652,2653,18566


## 7. Pesquisas levadas ao bairro (MRP) e o modelo ecológico

Cruzamentos publicados pelo Datafolha (nacional) e pela AtlasIntel (estaduais), extraídos dos relatórios e
conferidos por soma. Como só há marginais publicadas, o MRP é sintético (MrsP): um modelo aditivo em sexo, idade,
cor e região ou UF, pós-estratificado pela composição de cada área no Censo 2022. Renda, escolaridade e religião
ficam de fora da pós-estratificação, porque o Censo não as publica por bairro. Orientação sexual não tem dado oficial
por bairro nem por município e não é usada.

In [11]:
pd.read_csv("variantes/comparacoes/mrp_x_ei.csv")

,variante_mrp,unidades,correlacao_com_ei,dif_media_pp,dif_abs_mediana_pp,pct_divergencia_maior_10pp
0,R1,17592,0.776,3.51,9.70,48.6
1,R2,7731,0.719,2.89,9.47,47.7
2,R3,17592,0.747,3.54,10.28,51.3


In [12]:
pd.read_csv("variantes/comparacoes/mrp_x_ei_uf.csv").sort_values("div_media_pp")

,uf,div_media_pp,div_abs_mediana_pp
24,SE,-17.13,16.23
15,PE,-14.04,13.77
2,AM,-10.84,11.29
16,PI,-5.88,8.04
9,MA,-5.46,9.01
5,CE,-4.53,8.85
4,BA,-3.05,7.47
12,MT,-2.84,7.46
26,TO,-1.73,6.67
14,PB,-1.06,11.24


## 8. Perfil e temas por área

O perfil vem do Censo 2022 por setor (sexo, idade, cor ou raça, alfabetização e renda do responsável), somado
na área, e do contexto municipal (Novo CAGED, Bolsa Família e CadÚnico). Os temas são **hipóteses de comunicação**
derivadas de dados agregados: importância do tema nas pesquisas × perfil da área × existência de proposta com
fonte. Nenhum gatilho usa cor ou raça, gênero, religião ou orientação sexual. "Flávio mais fraco" = temas que o plano
de governo dele não menciona (salário mínimo, escala 6x1, isenção do IR, regra da Previdência), com fonte na matriz.

In [13]:
tm = pd.read_parquet("variantes/U4/temas_unidades.parquet")
tm["temas_aderentes_lula"].str.split(";").explode().value_counts().to_frame("áreas em que o tema está entre os 3 primeiros")

,áreas em que o tema está entre os 3 primeiros
temas_aderentes_lula,
saude_sus,18356
renda_salario_minimo,13549
programas_sociais,4682
educacao,4176
agricultura_familiar,3945
periferias_favelas,3564
soberania_nacional,2423
previdencia,1448
moradia_mcmv,1385


In [14]:
tm.merge(es[["id_unidade", "nivel_P2", "pot_p50"]], on="id_unidade").query("nivel_P2 == 7") \
  .sort_values("pot_p50", ascending=False)[["uf", "nome", "temas_aderentes_lula", "temas_flavio_mais_fraco", "temas_disputados", "sinais_fortes"]].head(15)

,uf,nome,temas_aderentes_lula,temas_flavio_mais_fraco,temas_disputados,sinais_fortes
18981,GO,Cidade Satélite São Luiz e arredores,saude_sus;educacao;renda_salario_minimo,saude_sus;jornada_6x1,seguranca_publica;corrupcao,
19092,GO,Residencial Monte Sião e arredores,periferias_favelas;educacao;saude_sus,periferias_favelas;saude_sus,seguranca_publica;corrupcao,favela
18980,GO,Jardim das Acácias e arredores,saude_sus;educacao;periferias_favelas,saude_sus;jornada_6x1,seguranca_publica;corrupcao,renda_media;jovens;baixa_alfabetizacao;favela
17773,SP,Jardim Aracati e arredores,periferias_favelas;educacao;moradia_mcmv,periferias_favelas;jornada_6x1,seguranca_publica;custo_de_vida_inflacao_alime...,baixa_alfabetizacao;favela
18985,GO,Residencial Alvino Albino e arredores,saude_sus;educacao;renda_salario_minimo,saude_sus;jornada_6x1,seguranca_publica;corrupcao,
266,MA,Centro e entorno,saude_sus;educacao;jornada_6x1,saude_sus;jornada_6x1,seguranca_publica;corrupcao,jovens
16401,MG,N.Sra. da Guia e arredores,saude_sus;periferias_favelas;moradia_mcmv,saude_sus;periferias_favelas,seguranca_publica;corrupcao,emprego_fraco;favela
19000,GO,Residencial Laurentino Martins e arredores,saude_sus;soberania_nacional;educacao,saude_sus;impostos_isencao_ir,seguranca_publica;corrupcao,soberania
18996,GO,Setor Bela Vista e arredores,saude_sus;educacao;jornada_6x1,jornada_6x1;saude_sus,seguranca_publica;corrupcao,jovens;emprego_fraco
17785,SP,Jardim Almeida e arredores,saude_sus;periferias_favelas;educacao,saude_sus;periferias_favelas,seguranca_publica;custo_de_vida_inflacao_alime...,jovens;baixa_alfabetizacao;favela


### 8.1 O que os planos de governo oficiais dizem, tema a tema

Os planos registrados no TSE (`Plano de governo/proposta-pt.pdf` e `proposta-pl.pdf`) foram segmentados em parágrafos
e classificados por um modelo de linguagem **local** (nada saiu da máquina), com conferência por código: tema e público
só valem com evidência textual no parágrafo, a meta numérica é detectada no texto, e a citação é sempre literal.
Força = compromissos concretos + 2 × metas numéricas.

In [15]:
pp = pd.read_csv("bairros/insumos/temas/propostas_por_tema.csv")
pp.pivot_table(index="tema", columns="candidato", values=["compromissos", "metas", "forca"], fill_value=0).astype(int)

compromissos       forca       metas     
candidato                              flavio lula flavio lula flavio lula
tema                                                                      
agricultura_familiar                        2    7      2    9      0    1
agronegocio_meio_ambiente                  23   39     27   49      2    5
corrupcao                                  11    3     11    5      0    1
costumes_religiao                          10    0     12    0      1    0
custo_de_vida_inflacao_alimentos           18   12     20   14      1    1
educacao                                   18   24     24   32      3    4
emprego                                    33   25     39   33      3    4
endividamento_apostas                       3    3      5    5      1    1
impostos_isencao_ir                        13    6     13   12      0    3
instituicoes_stf_reeleicao                  6    3      6    3      0    0
jornada_6x1                                 2    2      2    2      0    0
moradia_mcmv                                8    7     12   15      2    4
periferias_favelas                          1    7      3   15      1    4
previdencia                                 7    6      7    6      0    0
programas_sociais                           7    6      7   12      0    3
renda_salario_minimo                        0    7      0   11      0    2
saneamento                                  2    3      2    5      0    1
saude_sus                                  10   24     12   40      1    8
seguranca_publica                          13   16     23   22      5    3
soberania_nacional                          6   18      8   20      1    1
transporte_mobilidade                       9    6     11   10      1    2

In [16]:
pp[pp.candidato == "lula"][["tema", "citacao_pagina", "citacao"]].sort_values("tema").head(25)

,tema,citacao_pagina,citacao
21,agricultura_familiar,59,para nossa população.
22,agronegocio_meio_ambiente,79,Fortaleceremos a Inteligência de Estado como i...
23,corrupcao,27,"Com a Lei Antifacção, sancionada em 2026, ampl..."
24,costumes_religiao,40,A cultura é uma dimensão estratégica da nossa ...
25,custo_de_vida_inflacao_alimentos,64,A carteira do Novo PAC reúne R$ 595 bilhões em...
26,educacao,31,O Compromisso Nacional Criança Alfabetizada in...
27,emprego,73,Avançaremos na agenda ecológica com a mobiliza...
28,endividamento_apostas,39,intensificação da redução da mortalidade mater...
29,impostos_isencao_ir,19,Esse processo exige políticas públicas transve...
30,instituicoes_stf_reeleicao,16,e o orçamento secreto são práticas que mudaram...


### 8.2 Soberania nacional: onde o tema pode ter força

Não há pesquisa com recorte geográfico sobre o tema. Nacionalmente, 64% aprovam a defesa da soberania diante das tarifas
dos EUA (Genial/Quaest, set/2025), e 47% concordam mais com Lula do que com Flávio (35%) sobre o tarifaço
(Genial/Quaest, jun/2026). A força local é aproximada pela **exposição ao mercado dos EUA** (exportações de 2025 por
município, Comex Stat/MDIC) e pela **Amazônia Legal**. É uma hipótese: onde o tarifaço pesa na economia local, a defesa
da soberania tende a ressoar mais. O Comex atribui a exportação ao município do estabelecimento exportador (às vezes a
sede), e não desce ao bairro: no bairro, o sinal é o do município.

In [17]:
sb = pd.read_csv("bairros/insumos/contexto/soberania_municipios.csv")
print(sb["forca_soberania"].value_counts())
sb[sb.forca_soberania == "alta"].sort_values("exp_eua_usd", ascending=False).head(25)[
    ["municipio", "uf", "exp_eua_usd", "parcela_eua", "eua_por_eleitor", "sede_exportadora", "amazonia_legal"]]

forca_soberania
baixa    4411
média    1053
alta      107
Name: count, dtype: int64


,municipio,uf,exp_eua_usd,parcela_eua,eua_por_eleitor,sede_exportadora,amazonia_legal
27,SÃO JOSÉ DOS CAMPOS,SP,2.147147e+09,0.528644,3974.600523,False,False
25,PIRACICABA,SP,1.297704e+09,0.470150,4169.840405,False,False
40,SERRA,ES,1.105459e+09,0.595961,3055.974573,False,False
3,SÃO JOÃO DA BARRA,RJ,8.062218e+08,0.242498,18708.447324,True,False
4,SÃO GONÇALO DO AMARANTE,CE,7.913132e+08,0.651759,15408.687723,False,False
125,DUQUE DE CAXIAS,RJ,7.673453e+08,0.298571,1209.000378,False,False
0,GAVIÃO PEIXOTO,SP,7.222411e+08,0.619760,173407.235534,True,False
8,MATÃO,SP,6.540016e+08,0.447526,10262.872923,False,False
17,ARACRUZ,ES,4.658380e+08,0.297977,6201.416877,False,False
5,ILHABELA,SP,4.188974e+08,0.222409,13475.437335,True,False


In [18]:
import os
p = "variantes/U4/soberania_rm.csv"
pd.read_csv(p).head(15) if os.path.exists(p) else print("pendente: b14 para U4")

,nome_rm,aptos,exp_eua_usd,exp_total_usd,municipios_forca_alta,eua_por_eleitor,parcela_eua
0,RM do Vale do Paraíba e Litoral Norte (SP),1927778,3.129750e+09,1.011416e+10,3,1623.501316,0.309442
1,RM de Piracicaba (SP),1146431,1.695317e+09,4.908921e+09,3,1478.778008,0.345354
2,RM da Grande Vitória (ES),1422239,1.504775e+09,4.860306e+09,1,1058.032349,0.309605
3,RM de Jaraguá do Sul (SC),238339,2.399208e+08,1.002451e+09,1,1006.636656,0.239334
4,RM do Sudoeste Maranhense (MA),543264,4.603415e+08,1.101204e+09,3,847.362481,0.418035
5,RM de Jundiaí (SP),632857,4.080908e+08,1.809641e+09,1,644.838832,0.225509
6,RM do Rio de Janeiro (RJ),9554917,5.855680e+09,4.727856e+10,2,612.844677,0.123855
7,RM de Joinville (SC),665173,3.829956e+08,2.759476e+09,3,575.783458,0.138793
8,RM do Contestado (SC),416718,2.246460e+08,8.228576e+08,2,539.083898,0.273007
9,RM da Baixada Santista (SP),1447703,7.585366e+08,8.801432e+09,0,523.958705,0.086183


In [19]:
p = "variantes/U4/soberania_areas.csv"
if os.path.exists(p):
    a = pd.read_csv(p)
    if "coincide_baixo_esforco" in a:
        display(a[a.coincide_baixo_esforco.fillna(False).astype(bool)].sort_values("pot_p50", ascending=False)[["uf", "nome", "cd_mun_ibge", "nivel_P2", "pot_p50", "motivo"]].head(25))

,uf,nome,cd_mun_ibge,nivel_P2,pot_p50,motivo
19050,GO,Residencial Laurentino Martins e arredores,5208608,7.0,2990.54470,exportação aos EUA
19110,GO,Setor Central e entorno,5209101,7.0,1649.96940,exportação aos EUA
16841,MG,São Paulo e arredores,3147105,6.0,1060.95780,exportação aos EUA
12550,SP,Jardim Híipico e arredores,3512001,7.0,930.89856,exportação aos EUA
18987,MT,Centro Novo e entorno,5106422,7.0,643.62915,exportação aos EUA;Amazônia Legal
5955,ES,Fátima,3200607,7.0,614.52500,exportação aos EUA
13532,GO,Vila Bandeira Branca,5214002,7.0,610.58660,exportação aos EUA
1693,SP,Centro e entorno,3553807,7.0,607.35840,exportação aos EUA
17941,PR,Centro e entorno,4102901,7.0,510.65253,exportação aos EUA
1056,MG,Centro e entorno,3145901,7.0,471.67148,exportação aos EUA


## 9. Variantes lado a lado

In [20]:
pd.read_csv("variantes/comparacoes/metodo_MC_x_M2.csv")

,medida,correlacao,dif_mediana_abs,soma_mc,soma_m2,mudam_de_nivel,unidades,mc,m2
0,pot_p50,0.983,0.000000,1549800.0,1633616.0,NaN,NaN,NaN,NaN
1,margem_base_p50,1.000,10.055000,-6213117.0,-6197457.0,NaN,NaN,NaN,NaN
2,mob_p50,0.919,18.266001,-1121177.0,-1359625.0,NaN,NaN,NaN,NaN
3,lula2_base_p50,1.000,0.001000,NaN,NaN,NaN,NaN,NaN,NaN
4,prob_mob_rende,0.826,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
5,nivel_P1,0.860,NaN,NaN,NaN,9668.0,17139.0,NaN,NaN
6,nivel_P2,0.936,NaN,NaN,NaN,6294.0,17139.0,NaN,NaN
7,nivel_P3,0.743,NaN,NaN,NaN,11748.0,17139.0,NaN,NaN
8,largura_relativa_ic90_potencial,NaN,NaN,NaN,NaN,NaN,NaN,0.06,0.044


In [21]:
pd.read_csv("variantes/U4/MC/esforco_mudancas.csv")

,de,para,unidades,mudam_de_nivel,mudam_2_ou_mais,nivel7_em_comum,nivel7_de
0,P1,P2,18566,13089,6015,1751,2653
1,P1,P3,18566,10895,3772,1833,2653
2,P2,P3,18566,12915,6155,1672,2653


## 10. O que isto não diz

* **Não descreve pessoas.** Um bairro de nível 7 não tem "eleitores fáceis"; tem, no agregado, muito voto a recuperar
  por eleitor e um histórico de mobilização que tendeu a favorecer Lula.
* **Local de votação não é moradia.** As áreas descrevem quem vota nas escolas da área. O perfil do Censo descreve
  quem mora nela.
* **Falácia ecológica.** O risco cresce quanto menor a área. As áreas têm, na mediana, cerca de 4 mil eleitores.
* **Transferências de 2022.** A projeção supõe que eleitores de terceira via e quem não votou se comportam como em
  2022. A terceira via de 2026 é mais à direita: o cenário "direita" existe para isso.
* **Pesquisas.** São marginais publicadas por dois institutos. O MRP ignora renda, escolaridade e religião, que o
  Censo não traz por bairro. O 1º turno mostrou erro das pesquisas a favor de Lula.
* **Temas.** São hipóteses de comunicação, não diagnóstico. Não dependem de raça, gênero ou religião.
* **Os níveis são triagem.** Mudam com os pesos e com o método (seção 9).

## 11. Reprodução

```bash
# ambiente (Python 3.12)
uv venv .venv-bairros --python 3.12 && uv pip install --python .venv-bairros/bin/python -r requirements-bairros.txt
P=.venv-bairros/bin/python
$P bairros/baixar.py tudo                     # TSE e IBGE (~15 GB), com retomada
$P bairros/b01_secoes.py 2026 2022 2018 2014 2010
$P bairros/b02_locais.py 2026 2022 2018 2014 2010
$P bairros/b03_geocodificar.py 2026 2022
$P bairros/b04a_censo_setores.py
$P bairros/b04_unidades.py U1 U2 U3
$P bairros/rodar_ei.py U4 --paralelo 3          # MCMC por UF (checkpoints em variantes/U4/MC)
$P bairros/b05_ei_bairros.py --unidade U4 --metodo M2 && $P bairros/b05b_m2_por_uf.py --unidade U4
$P bairros/b06_montecarlo_bairros.py --unidade U4 --metodo MC --sorteios 10000
$P bairros/b08_esforco.py --unidade U4 --metodo MC
$P bairros/b07a_pesquisas.py && $P bairros/b07b_mrp.py --unidade U4
$P bairros/b09_temas.py --unidade U4
$P bairros/b11_comparar.py
$P mapa/construir_mapa_bairros.py --unidade U4 --metodo MC --pesos P2
$P construir_notebook_bairros.py --unidade U4 --metodo MC --pesos P2
```

Sementes fixas em todos os scripts. Versões dos pacotes em `requirements-bairros.txt`. O MCMC não é bit a bit
reprodutível entre execuções nesta máquina: a checagem de cadeias presas e os diagnósticos ficam
registrados em `variantes/*/MC/diag_*.json`.

**Fontes:** TSE (Dados Abertos), IBGE (Censo 2022: malhas e agregados por setor; RMs), malhas
de bairros de prefeituras, MTE (Novo CAGED), MDS (Bolsa Família, CadÚnico), Datafolha e AtlasIntel (pesquisas
registradas no TSE).